<a href="https://colab.research.google.com/github/Mithun-Mh/Simple-RAG-LangChain-Project/blob/main/Simple_RAG_Application_%5BLangChain_and_Gemini%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Simple RAG Application using LangChain and OpenAI**



In [ ]:
import os
from google.colab import userdata

In [ ]:
%pip install -q langchain langchain-google-genai langchain-chroma langchain-community

Initialize Gemini LLM

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=0
)

Initialize Embedding Model

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.environ.get("GOOGLE_API_KEY")
)

Create and Embed Documents

In [ ]:

from langchain_core.documents import Document

# Define a list of documents with content and metadata
documents = [
    Document(
        page_content="The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.",
        metadata={"source": "cricket news"},
    ),
    Document(
        page_content="The world of football is buzzing with excitement as major tournaments and league matches continue to captivate fans globally.In the UEFA Champions League, the semi-final matchups have been set, with defending champions Real Madrid set to face Manchester City, while Bayern Munich will take on Paris Saint-Germain.Both ties promise thrilling encounters, featuring some of the best talents in world football.",
        metadata={"source": "football news"},
    ),
    Document(
        page_content="As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives.",
        metadata={"source": "election news"},
    ),
    Document(
        page_content="The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.",
        metadata={"source": "ai revolution news"},
    ),
]

In [ ]:
# Create a vector store using the documents and embedding model
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embedding=embedding_model,
)

Perform Similarity Search

In [ ]:
results = vectorstore.similarity_search("machine learning")

for result in results:
  print("------------------------")
  print(result.page_content)
  print(result.metadata)

------------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
------------------------
As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives.
{'source': 'election news'}
------------------------
The T20 World Cup 2024 is in full swing, bringi

Embed Query and Perform Similarity Search by Vector

In [ ]:
# Embed a query using the embedding model
query_embedding = embedding_model.embed_query("machine learning")

query_embedding[:10]

[-0.006826791,
 0.0032323739,
 0.0036080698,
 -0.063535206,
 -0.016612697,
 0.003850432,
 0.0019643204,
 0.027494766,
 0.0015598271,
 0.012391397]

In [ ]:
# Print the length of the query embedding
len(query_embedding)

3072

In [ ]:
results = vectorstore.similarity_search_by_vector(query_embedding)

for result in results:
  print("------------------------")
  print(result.page_content)
  print(result.metadata)

------------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
------------------------
As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives.
{'source': 'election news'}
------------------------
The T20 World Cup 2024 is in full swing, bringi

Create Retriever

In [ ]:

# Create a retriever from the vector
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

# Perform batch retrieval using the retriever
batch_results = retriever.batch(["machine learning","test match"])

for result in batch_results:
  print("------------------------")
  for doc in result:
    print(doc.page_content)
    print(doc.metadata)

------------------------
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
------------------------
The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.
{'source': 'cricket news'}


Create Prompt Template

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define a message template for the chatbot
message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

# Create a chat prompt template from the message
prompt = ChatPromptTemplate.from_messages([("human", message)])

Chain Retriever and Prompt Template with LLM

In [ ]:
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# Create a retrieval chain that extracts the 'input' string for the retriever
retrieval_chain = (
    {
        "context": itemgetter("input") | retriever,
        "input": itemgetter("input")
    }
    | document_chain
)

# Invoke with the dictionary
response = retrieval_chain.invoke({"input": "current state of 2024 t20 world cup"})
print(response)

Based on the provided context, the 2024 T20 World Cup is currently "in full swing," bringing excitement and drama to fans worldwide. 

Key details about its current state include:
* **Ongoing matches and preparations:** Teams are actively competing, such as India (captained by Rohit Sharma) preparing for a crucial match against Ireland.
* **Controversy:** The tournament has already faced controversy regarding pitch conditions at the Nassau County International Cricket Stadium in New York, which received criticism following a low-scoring match between Sri Lanka and South Africa.


In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using the provided context.

Context:
{context}

Question:
{input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [ ]:
response = llm.invoke("Explain RAG in two sentences.")
print(response.content)

[{'type': 'text', 'text': 'Retrieval-Augmented Generation (RAG) is an AI technique that connects a language model to external data sources to retrieve relevant information before answering a prompt. By grounding the model in this newly fetched context, RAG generates more accurate, up-to-date responses and significantly reduces hallucinations.', 'extras': {'signature': 'Eu0OCuoOAWkUfRPFiAWJZQ/X2fxxn3iTEmH6F0Q3xqdNCjgjYKkqxcNrp/6769vqWZaKbD0VHyyZYgTqN6DUeDxaOrzCSxteOdPBxCpzr3Zso4+U47Hqyj+xVB3ld8pNDD4ZasM/+FQWH4wPbXI4u+T7mdQgARNyDWh+bddX4dZubK3X6JR0NtMcYGtH15bc/18Lht1RBbsIcVkrfCwlhMXk3lWXwgEwUjeS/Eb11MVQ5IG6HvFBLtkiOOs5qCTyHLbXhFynrYSQndUEtFoty6IzgBIVbmwSQW5ADlp6HMljAUlbCsM+BTOnI25zVYtqzSTkgo68wu9Ao3gjhmqRfRwaas+8SR5rZphnTQPsqokiers2k+i1iHjVhISL/KULDq7t5ixdIuckmDZ0KH+zzBmYbCXAP1y0J83P+9fFZwmst6E+nEPocBT4PmcWA0Bd25ZpaPubpk3e8TVW9i6gL0pUZKX4dyOSYxRaCMZ6lY++CHRkSVc2J8Jdep0mUQTA6EsnCaXZmZpNq/v5QqzxmPyy/en2624BbFB5b8fTcfmjfd6E07pHPl3fD+bauMxmhseX140E2bm/EKNUPI0rBPSbqYpzy1LxpNU/qu8iKJmxd0xGX+CQ